# Timeline Event Ingestion Test (OxCGRT Finland)

This notebook validates the separate timeline event source layer:
- remote fetch (only for ingest/refresh)
- local raw snapshot save
- local processed normalized save
- reload from processed local file
- TimelineEvent object creation and inspection

Note: normal project usage should read local processed files, not fetch every run.


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "extensions" / "scenario_api").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate extensions/scenario_api from current working directory")

extensions_dir = repo_root / "extensions"
if str(extensions_dir) not in sys.path:
    sys.path.insert(0, str(extensions_dir))

from scenario_api.timeline import (
    load_timeline_events_from_processed,
    filter_timeline_events,
    timeline_events_to_table,
)
from scenario_api.timeline_sources import (
    download_and_save_oxcgrt_finland_timeline,
    load_processed_timeline_events_table,
)

import pandas as pd
import matplotlib.pyplot as plt

## B) Configure paths

In [ ]:
raw_path = repo_root / "data" / "raw" / "oxcgrt_finland_2020_2022_raw.csv"
processed_path = repo_root / "data" / "processed" / "oxcgrt_finland_2020_2022_timeline.csv"

raw_path, processed_path

## C) Fetch source + save local snapshots

In [ ]:
# This cell fetches remote data. After first successful run, it can be commented out.
saved_raw_path, saved_processed_path = download_and_save_oxcgrt_finland_timeline(
    raw_output_path=raw_path,
    processed_output_path=processed_path,
    start_date="2020-01-01",
    end_date="2022-12-31",
)

print("Saved raw snapshot:", saved_raw_path)
print("Saved processed timeline:", saved_processed_path)

## D) Reload processed local file

In [ ]:
processed_table = load_processed_timeline_events_table(processed_path)
processed_table.head(10)

## E) Create TimelineEvent objects

In [ ]:
events = load_timeline_events_from_processed(processed_path)

print("Number of events:", len(events))
print("Min date:", min(e.date for e in events))
print("Max date:", max(e.date for e in events))
print("Distinct event types:", sorted(set(e.event_type for e in events)))

## F) Inspect events

In [ ]:
print("First 10 events:")
for e in events[:10]:
    print(e)

facial_events = filter_timeline_events(events, event_type="facial_coverings")
testing_events = filter_timeline_events(events, event_type="testing_policy")

print("\nFacial covering events (first 10):")
for e in facial_events[:10]:
    print(e)

print("\nTesting policy events (first 10):")
for e in testing_events[:10]:
    print(e)

## G) Summary + simple plot

In [ ]:
events_table = timeline_events_to_table(events)

summary = (
    events_table.groupby("event_type", as_index=False)
    .size()
    .rename(columns={"size": "n_events"})
    .sort_values("n_events", ascending=False)
)

display(summary)

# Step plot for one event type.
plot_df = events_table[events_table["event_type"] == "facial_coverings"].copy()
plot_df["date"] = pd.to_datetime(plot_df["date"])
plot_df = plot_df.sort_values("date")

fig, ax = plt.subplots(figsize=(10, 4))
ax.step(plot_df["date"].to_numpy(), plot_df["value"].to_numpy(), where="post", linewidth=2)
ax.set_title("OxCGRT Finland: facial_coverings (2020-2022)")
ax.set_xlabel("Date")
ax.set_ylabel("Value")
ax.grid(alpha=0.3)
plt.show()